# Deepfake Detection — Training on Google Colab

**Before running:** Make sure Runtime → Change runtime type → **T4 GPU** is selected.

This notebook:
1. Mounts Google Drive
2. Installs dependencies
3. Runs the full training pipeline
4. Saves the best model + all plots back to Drive

## Step 1 — Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name    :', torch.cuda.get_device_name(0))
    print('VRAM        :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## Step 2 — Mount Drive & Set Project Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ⚠️ CHANGE THIS to the path of your project folder on Google Drive
PROJECT_DIR = '/content/drive/MyDrive/deepfake-detection'

import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

## Step 3 — Install Dependencies

In [ ]:
# Colab already has torch, torchvision, numpy, pandas, matplotlib, seaborn, scikit-learn
# We just need the extra packages
!pip install -q timm albumentations pytorch-grad-cam facenet-pytorch
print('Dependencies installed ✓')

## Step 4 — Verify Dataset

In [ ]:
import yaml
from src.dataset import load_config, collect_image_paths

cfg = load_config('config.yaml')
d   = cfg['data']

for split, rdir, fdir in [
    ('Train', d['train_real'], d['train_fake']),
    ('Val',   d['val_real'],   d['val_fake']),
    ('Test',  d['test_real'],  d['test_fake']),
]:
    r = collect_image_paths(rdir)
    f = collect_image_paths(fdir)
    print(f'{split:<6} → Real: {len(r):>7,}  Fake: {len(f):>7,}  Total: {len(r)+len(f):>8,}')

print('\nDataset verified ✓')

## Step 5 — Run Training

In [ ]:
# This runs the full training pipeline:
# - Freeze backbone 2 epochs → unfreeze → full fine-tune
# - Early stopping (patience=4)
# - Best model saved to checkpoints/best_model.pth
# - Plots saved to results/plots/
# - Test set evaluation at the end

from src.train import train
history, best_auc = train('config.yaml')
print(f'\nTraining complete! Best Val AUC: {best_auc:.4f}')

## Step 6 — Generate Grad-CAM Visualizations

In [ ]:
import torch
from src.dataset import build_dataloaders
from src.model import build_model, load_checkpoint
from src.gradcam import generate_gradcam_grid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = build_model(cfg, device)
load_checkpoint(model, cfg['paths']['best_model'], device=device)

_, _, test_loader, _ = build_dataloaders(cfg, verbose=False)

generate_gradcam_grid(model, test_loader, cfg, device, n_samples=16)
print('Grad-CAM visualizations saved ✓')

## Step 7 — View Results

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Print final metrics
with open('results/metrics/test_metrics.json') as f:
    m = json.load(f)

print('='*45)
print('  FINAL TEST SET RESULTS')
print('='*45)
print(f"  AUC-ROC   : {m['auc']:.4f}")
print(f"  EER       : {m['eer']:.4f}  ({m['eer']*100:.2f}%)")
print(f"  Accuracy  : {m['accuracy']:.4f}  ({m['accuracy']*100:.2f}%)")
print(f"  F1-Score  : {m['f1']:.4f}")
print(f"  Precision : {m['precision']:.4f}")
print(f"  Recall    : {m['recall']:.4f}")
print('='*45)

In [ ]:
# Display all result plots inline
plots = [
    ('results/plots/training_curves.png',    'Training Curves'),
    ('results/plots/roc_curve.png',          'ROC Curve'),
    ('results/plots/confusion_matrix.png',   'Confusion Matrix'),
    ('results/plots/benchmark_comparison.png','Benchmark Comparison'),
    ('results/plots/score_distribution.png', 'Score Distribution'),
    ('results/gradcam/gradcam_grid.png',     'Grad-CAM Visualizations'),
]

for path, title in plots:
    if os.path.exists(path):
        fig, ax = plt.subplots(figsize=(12, 7))
        ax.imshow(mpimg.imread(path))
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'Not found: {path}')

## Step 8 — Download Model to Local Machine

Since the project is in Google Drive, your trained model is already saved at:
```
deepfake-detection/checkpoints/best_model.pth
```
Just download it from Drive to your local `checkpoints/` folder.

In [ ]:
# Optional: also download directly from Colab to your browser
from google.colab import files
files.download('checkpoints/best_model.pth')
print('Model download started — check your browser downloads')